# `candidate_final.csv`: the complete pipeline in one notebook

This notebook contains **all the code** needed to produce `candidate_final.csv`. It needs only `train.csv` and `test.csv` (plus, optionally, `candidate_v4.csv`; see step 1). It replaces these files:

| Replaced file | What it contributed |
|---|---|
| `energy_model.py` | v4: per-missing-pattern models, the v3 notebook model, test-mix weights |
| `experiments/round2_models.py`, `experiments/notebook_baseline.py` | the new model's Ridge + trees and expected-value features |
| `final_model.py`, `final_robust.py` | LightGBM imputers (instead of CatBoost), 10-seed averaging, final blend |
| `model_config.json`, `model_v5_config.json` | settings (written into the cells below) |

**The final prediction**

$$\text{prediction} = 0.3 \times \text{v4} + 0.7 \times \text{average of 10 seeds of the LightGBM-imputed model}$$

**Ground rules:** no test answers are used anywhere. Test *inputs* are used without labels, as unlabelled data: to train the imputers, to copy realistic blank patterns, and to weight training rows toward the test set's building/event mix.

**Environment:** run with the platform's library versions (`requirements-image.txt`), for example the `datathon-platform-env` environment. Full run time is about 4–5 minutes.

In [1]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.experimental import enable_iterative_imputer  # must come before importing IterativeImputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits
import lightgbm as lgb
import sklearn

warnings.filterwarnings('ignore')
ROOT = Path('.')                      # folder with train.csv and test.csv
OUTPUT = ROOT / 'candidate_final.csv'
REFIT_V4 = False                      # False: use candidate_v4.csv if it exists (exact file that scored 3.37); True: retrain v4
SEEDS = tuple(range(527, 537))        # 10 seeds for the LightGBM-imputed model
WEIGHT_NEW = 0.70                     # weight on the LightGBM-imputed model; the rest goes to v4

train = pd.read_csv(ROOT / 'train.csv')
test = pd.read_csv(ROOT / 'test.csv')
print('scikit-learn', sklearn.__version__, '| lightgbm', lgb.__version__, '| pandas', pd.__version__)
print(f'train {len(train):,} rows | test {len(test):,} rows | test blanks: {test[["temperature", "humidity", "occupancy", "previous_usage"]].isna().sum().to_dict()}')

scikit-learn 1.5.2 | lightgbm 4.5.0 | pandas 2.2.3
train 8,000 rows | test 3,000 rows | test blanks: {'temperature': 211, 'humidity': 183, 'occupancy': 206, 'previous_usage': 203}


## Shared definitions and the test-mix weights

`shift_weights` gives each training row a weight, so the training data's mix of buildings and **unusual event rows** looks like the test set's. The six events are heat, rain, occupancy surge or drop, and `previous_usage` spike or drop, each judged against the typical value for that building, hour and weekday/weekend. It looks only at test *inputs*. The test set has about five times as many event rows as train.

This cell also defines the per-pattern models used in v4 (`features`, `PatternModel`). The code is copied unchanged from `energy_model.py`.

In [2]:
NUM = ['temperature', 'humidity', 'occupancy', 'previous_usage']
BUILDINGS = ['ADM_A','BUS_A','BUS_B','ENG_A','ENG_B','LEC_A','LIB_A','RES_A','RES_B','SCI_A','SCI_B','SPT_A']
DAYS = {'Monday':0,'Tuesday':1,'Wednesday':2,'Thursday':3,'Friday':4,'Saturday':5,'Sunday':6}

def features(df, observed, harmonics=1, extra='none'):
    z = {c:df[c].to_numpy(float) for c in observed}
    w = df.day_of_week.isin(['Saturday','Sunday']).to_numpy(float)
    h = df.hour.to_numpy(float)
    z['weekend'] = w
    for k in range(1, harmonics+1):
        z[f'sin{k}'] = np.sin(2*np.pi*k*h/24)
        z[f'cos{k}'] = np.cos(2*np.pi*k*h/24)
    z['month_sin'] = np.sin(2*np.pi*df.month.to_numpy()/12)
    z['month_cos'] = np.cos(2*np.pi*df.month.to_numpy()/12)
    if extra in ['weekend','interactions','hinges']:
        z['weekend_sin'] = w*z['sin1']
        z['weekend_cos'] = w*z['cos1']
    if extra in ['interactions','hinges']:
        if 'occupancy' in observed:
            z['occ_weekend'] = z['occupancy']*w
            z['occ_sin'] = z['occupancy']*z['sin1']
            z['occ_cos'] = z['occupancy']*z['cos1']
    if extra == 'hinges':
        if 'temperature' in observed:
            z['cooling'] = np.maximum(0,z['temperature']-28)
        if 'humidity' in observed:
            z['rain'] = np.maximum(0,z['humidity']-90)
    return np.column_stack(list(z.values()))

class PatternModel:
    """Train one supervised model for each subset of available numeric inputs."""
    def __init__(self, harmonics=1, alpha=10., extra='none', tree_weight=0., trees=180, min_leaf=35, weight_power=0., tree_leaves=12, categorical=False):
        self.harmonics=harmonics;self.alpha=alpha;self.extra=extra
        self.tree_weight=tree_weight;self.trees=trees;self.min_leaf=min_leaf;self.weight_power=weight_power;self.tree_leaves=tree_leaves;self.categorical=categorical
    def fit(self, df, patterns=None, test_features=None):
        self.models={}
        weights = shift_weights(df, test_features) ** self.weight_power if self.weight_power else np.ones(len(df))
        weights = pd.Series(weights / np.mean(weights), index=df.index)
        if patterns is None: patterns=range(16)
        for pattern in patterns:
            obs=[c for k,c in enumerate(NUM) if not (pattern & (1<<k))]
            sub=df.dropna(subset=obs).copy()
            a=features(sub,obs,self.harmonics,self.extra)
            sc=StandardScaler().fit(a);a=sc.transform(a)
            b=(sub.building_id.to_numpy()[:,None]==np.array(BUILDINGS)[None,:]).astype(float)
            x=np.column_stack([a,b,(a[:,:,None]*b[:,None,:]).reshape(len(a),-1)])
            sw = weights.loc[sub.index].to_numpy()
            m=Ridge(alpha=self.alpha).fit(x,sub.energy_usage,sample_weight=sw)
            t=None
            if self.tree_weight:
                tx=self.tree_features(sub,obs)
                t=HistGradientBoostingRegressor(max_iter=self.trees,max_leaf_nodes=self.tree_leaves,learning_rate=.05,
                      categorical_features=([False]*(tx.shape[1]-1)+[True]) if self.categorical else None,
                      min_samples_leaf=self.min_leaf,l2_regularization=8.,early_stopping=False,random_state=42)
                t.fit(tx,sub.energy_usage.to_numpy()-m.predict(x),sample_weight=sw)
            self.models[pattern]=(obs,sc,m,t)
        return self
    @staticmethod
    def tree_features(df,obs):
        a=df[['hour','month']+obs].to_numpy(float)
        return np.column_stack([a,df.day_of_week.map(DAYS),df.building_id.map(dict(zip(BUILDINGS,range(12))))])
    def predict(self,df):
        patterns=(df[NUM].isna().to_numpy().astype(int)*(2**np.arange(4))).sum(axis=1)
        pred=np.empty(len(df))
        for pattern in np.unique(patterns):
            ix=np.flatnonzero(patterns==pattern);sub=df.iloc[ix]
            obs,sc,m,t=self.models[pattern]
            a=sc.transform(features(sub,obs,self.harmonics,self.extra))
            b=(sub.building_id.to_numpy()[:,None]==np.array(BUILDINGS)[None,:]).astype(float)
            x=np.column_stack([a,b,(a[:,:,None]*b[:,None,:]).reshape(len(a),-1)])
            pred[ix]=m.predict(x)
            if t is not None:pred[ix]+=self.tree_weight*t.predict(self.tree_features(sub,obs))
        return pred

def event_groups(df, reference):
    """Feature-only event strata; typical values are fitted on the reference fold."""
    key=['building_id','hour','weekend']
    ref=reference.assign(weekend=reference.day_of_week.isin(['Saturday','Sunday']))
    typical=ref.groupby(key)[['occupancy','previous_usage']].median()
    q=df.assign(weekend=df.day_of_week.isin(['Saturday','Sunday'])).join(typical.add_suffix('_typical'),on=key)
    flags=np.column_stack([
        q.temperature>=33.5, q.humidity>=93,
        q.occupancy-q.occupancy_typical>=100, q.occupancy-q.occupancy_typical<=-60,
        q.previous_usage/q.previous_usage_typical>1.6, q.previous_usage/q.previous_usage_typical<.5])
    # A primary event class avoids unreliable weights for nearly empty intersections.
    event=np.where(flags.any(axis=1),np.argmax(flags,axis=1)+1,0)
    return pd.Series(event,index=df.index)

def shift_weights(source,target):
    if target is None: raise ValueError('Unlabelled test features required for shift weighting.')
    es=event_groups(source,source);et=event_groups(target,source)
    counts_s=pd.crosstab(source.building_id,es).reindex(index=BUILDINGS,columns=range(7),fill_value=0)
    counts_t=pd.crosstab(target.building_id,et).reindex(index=BUILDINGS,columns=range(7),fill_value=0)
    # Smooth event proportions toward the global event mix within each building.
    ps=(counts_s+15*(np.bincount(es,minlength=7)+1)/(len(es)+7)).div(counts_s.sum(axis=1)+15,axis=0)
    pt=(counts_t+15*(np.bincount(et,minlength=7)+1)/(len(et)+7)).div(counts_t.sum(axis=1)+15,axis=0)
    ratio=pt/ps
    b_ratio=(counts_t.sum(axis=1)/len(target))/(counts_s.sum(axis=1)/len(source))
    return np.clip(np.array([ratio.loc[b,e]*b_ratio[b] for b,e in zip(source.building_id,es)]),.2,10.)

## Step 1: v4 (scored 3.37 on its own)

v4 combines two models:

- **The v3 notebook model (25%).** MissForest-style filling (ExtraTrees), fitted on training rows only. Then a Ridge model with per-building slopes, event features and occupancy by part of day, blended 80/20 with gradient boosting on the change from `previous_usage`.
- **Per-missing-pattern models (75%).** One Ridge regression for each of the 16 combinations of available numeric inputs, with per-building effects and three daily sine/cosine curves, plus a small tree correction. Training rows are weighted toward the test mix. A row missing `previous_usage` is predicted directly by a model that never uses it, instead of from a guessed value.

The v3 code below is copied unchanged from `energy_model.py`.

In [3]:
num_cols = ['temperature', 'humidity', 'occupancy', 'previous_usage']
building_ids = ['ADM_A','BUS_A','BUS_B','ENG_A','ENG_B','LEC_A','LIB_A','RES_A','RES_B','SCI_A','SCI_B','SPT_A']
building_types = ['Administration','Business','Engineering','LectureHall','Library','Residential','Science','Sports']
day_to_int = {'Monday': 0, 'Tuesday': 1, 'Wednesday': 2, 'Thursday': 3,
              'Friday': 4, 'Saturday': 5, 'Sunday': 6}

def imputer_inputs(df):
    """The 4 columns that have blanks, plus never-blank columns the imputer can learn from."""
    X = df[num_cols].copy()
    X['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    X['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    X['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    X['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    X['is_weekend'] = df['day_of_week'].isin(['Saturday', 'Sunday']).astype(float)
    bid = pd.get_dummies(pd.Categorical(df['building_id'], categories=building_ids), prefix='bid').astype(float)
    bid.index = df.index
    return pd.concat([X, bid], axis=1)


def fit_imputer(train_df):
    """MissForest-style: each blank column is guessed by a tree model trained on the other columns."""
    imputer = IterativeImputer(
        estimator=ExtraTreesRegressor(n_estimators=100, min_samples_leaf=3, max_features=0.6,
                                      n_jobs=4, random_state=0),
        max_iter=8, random_state=0)
    imputer.fit(imputer_inputs(train_df))
    return imputer


def fill_blanks(imputer, df):
    df = df.copy()
    df[num_cols] = imputer.transform(imputer_inputs(df))[:, :len(num_cols)]
    return df


def linear_features(df, typical):
    X = df[num_cols].copy()
    X['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    X['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    X['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    X['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    X['is_weekend'] = df['day_of_week'].isin(['Saturday', 'Sunday']).astype(float)
    X['occ_x_weekend'] = X['occupancy'] * X['is_weekend']

    bid = pd.get_dummies(pd.Categorical(df['building_id'], categories=building_ids), prefix='bid').astype(float)
    btype = pd.get_dummies(pd.Categorical(df['building_type'], categories=building_types), prefix='bt').astype(float)
    bid.index = btype.index = df.index

    # each building gets its own slope for the main drivers
    inter = {}
    for col in ['occupancy', 'previous_usage', 'temperature', 'hour_sin', 'hour_cos']:
        for b in bid.columns:
            inter[f'{col}_x_{b}'] = X[col] * bid[b]
    for col in ['is_weekend', 'occ_x_weekend']:
        for b in btype.columns:
            inter[f'{col}_x_{b}'] = X[col] * btype[b]
        # NEW: how unusual the row is for its building, hour and weekday/weekend (event rows)
    key = pd.DataFrame({'building_id': df['building_id'], 'hour': df['hour'], 'is_weekend': X['is_weekend'] > 0})
    t = key.join(typical, on=['building_id', 'hour', 'is_weekend']).fillna(typical.median())
    events = pd.DataFrame({
        'occ_gap': X['occupancy'] - t['occupancy'],
        'prev_ratio': X['previous_usage'] / t['previous_usage'],
        'prev_gap': X['previous_usage'] - t['previous_usage'],
        'heat': (X['temperature'] >= 33.5).astype(float),
        'rain': (X['humidity'] >= 93).astype(float),
    }, index=df.index)
    for col in events.columns:
        for b in btype.columns:
            inter[f'{col}_x_{b}'] = events[col] * btype[b]

    # NEW: occupancy effect by part of the day, per building type
    part = pd.cut(df['hour'], [-1, 5, 8, 11, 14, 17, 20, 23], labels=False)
    daypart = pd.get_dummies(pd.Categorical(part, categories=range(7)), prefix='dp').astype(float)
    daypart.index = df.index
    for p in daypart.columns:
        for b in btype.columns:
            inter[f'occ_{p}_x_{b}'] = X['occupancy'] * daypart[p] * btype[b]

    return pd.concat([X, btype, bid, events, pd.DataFrame(inter, index=df.index)], axis=1)


def tree_features(df):
    X = df[['hour', 'month'] + num_cols].copy()
    X['day'] = df['day_of_week'].map(day_to_int)
    X['is_weekend'] = (X['day'] >= 5).astype(int)
    X['building_type'] = pd.Categorical(df['building_type'], categories=building_types)
    X['building_id'] = pd.Categorical(df['building_id'], categories=building_ids)
    return X


class NotebookBaseline:
    def fit(self, train_df):
        self.imputer = fit_imputer(train_df)
        train_df = fill_blanks(self.imputer, train_df)
        y = train_df['energy_usage']
        weekend = train_df.day_of_week.isin(['Saturday', 'Sunday'])
        self.typical = train_df.assign(is_weekend=weekend).groupby(
            ['building_id', 'hour', 'is_weekend'])[['occupancy', 'previous_usage']].median()
        self.ridge = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 13)))
        self.ridge.fit(linear_features(train_df, self.typical), y)
        self.hgb = HistGradientBoostingRegressor(
            categorical_features='from_dtype', max_iter=1500, learning_rate=.03,
            max_leaf_nodes=15, min_samples_leaf=30, l2_regularization=1.,
            early_stopping=True, n_iter_no_change=50, random_state=42)
        self.hgb.fit(tree_features(train_df), y-train_df.previous_usage)
        return self
    def predict(self, pred_df):
        df = fill_blanks(self.imputer, pred_df)
        ridge = self.ridge.predict(linear_features(df, self.typical))
        hgb = self.hgb.predict(tree_features(df)) + df.previous_usage.to_numpy()
        return .8*ridge + .2*hgb

In [4]:
PATTERN_CONFIG = {'harmonics': 3, 'alpha': 1, 'extra': 'none', 'tree_weight': 0.65, 'weight_power': 0.5, 'trees': 250, 'min_leaf': 25, 'tree_leaves': 15, 'categorical': True}   # from model_config.json


def v4_predictions(train, test):
    """v4 = 0.25 x v3 notebook model + 0.75 x per-missing-pattern models (as in energy_model.EnergyModel)."""
    inputs = test.drop(columns=['energy_usage'], errors='ignore')
    baseline = NotebookBaseline().fit(train)
    experts = PatternModel(**PATTERN_CONFIG).fit(train, test_features=inputs)
    return 0.25 * baseline.predict(inputs) + 0.75 * experts.predict(inputs)


v4_file = ROOT / 'candidate_v4.csv'
if not REFIT_V4 and v4_file.exists():
    v4_df = pd.read_csv(v4_file)
    assert v4_df['id'].equals(test['id'])
    v4 = v4_df['prediction'].to_numpy()
    print('v4: using candidate_v4.csv (the exact predictions that scored 3.37)')
else:
    with threadpool_limits(limits=4):
        v4 = v4_predictions(train, test)
    print('v4: retrained from scratch')

v4: using candidate_v4.csv (the exact predictions that scored 3.37)


**Why there's a `REFIT_V4` switch:** `candidate_v4.csv` was produced in an environment with newer library versions. Retrained here with the platform versions, v4 differs from that file by about 0.1 on average per row. To reproduce `candidate_final.csv` exactly, leave `REFIT_V4 = False`. Set it to `True` to build everything from `train.csv` and `test.csv` alone.

## Step 2: the LightGBM-imputed model (the main ingredient)

**2a. Imputers.** For each numeric input, a LightGBM model learns to predict it from the other inputs, plus building, building type and calendar. Three details:

- They learn from the input values of **train and test rows**, never from `energy_usage`.
- Each observed row is copied **3 more times with blank patterns drawn from real test rows**. The column being learned always stays filled in, so the imputers learn to cope with several blanks at once.
- Rows are weighted toward the test mix (`shift_weights` raised to the power 1.5).

These LightGBM imputers replace the CatBoost ones from `candidate_v5.csv`, because CatBoost isn't on the platform list. On held-out rows they scored as well or better.

**2b. Energy model.** Blanks are filled with the imputers' predictions. A Ridge model then uses the v3 features plus the imputers' **expected** `previous_usage` and `occupancy` for each row (also per building type). It's blended 80/20 with gradient boosting on the change from `previous_usage`.

In [5]:
NEW_MODEL_CONFIG = {'depth': 5, 'iterations': 1100, 'augment': 3, 'linear_mode': 'notebook', 'distill': True, 'tree_blend': 0.2, 'imputer_weight': 1.5}   # from model_v5_config.json
TYPES = building_types
CATEGORIES = {'building_id': BUILDINGS, 'building_type': TYPES, 'day_of_week': list(DAYS)}


def cat_features(df):
    """Inputs for the imputers (from experiments/round2_models.py)."""
    x = df[['building_id', 'building_type', 'hour', 'day_of_week', 'month'] + NUM].copy()
    x['weekend'] = df.day_of_week.isin(['Saturday', 'Sunday']).astype(int)
    x['hour_sin'] = np.sin(2 * np.pi * df.hour / 24); x['hour_cos'] = np.cos(2 * np.pi * df.hour / 24)
    x['month_sin'] = np.sin(2 * np.pi * df.month / 12); x['month_cos'] = np.cos(2 * np.pi * df.month / 12)
    return x


def as_categories(x):
    x = x.copy()
    for col, cats in CATEGORIES.items():
        x[col] = pd.Categorical(x[col], categories=cats)
    return x


class LightGBMImputer:
    def __init__(self, seed):
        self.model = lgb.LGBMRegressor(n_estimators=2 * NEW_MODEL_CONFIG['iterations'], learning_rate=0.025,
                                       max_depth=NEW_MODEL_CONFIG['depth'], num_leaves=2 ** NEW_MODEL_CONFIG['depth'],
                                       min_child_samples=30, reg_lambda=15.0, subsample=0.8, subsample_freq=1,
                                       colsample_bytree=0.8, cat_smooth=20, verbose=-1, random_state=seed, n_jobs=4)

    def fit(self, x, y, sample_weight):
        self.model.fit(as_categories(x), y, sample_weight=sample_weight)
        return self

    def predict(self, x):
        return self.model.predict(as_categories(x))


class LightGBMImputedModel:
    def __init__(self, depth=5, iterations=1100, augment=3, linear_mode='notebook', distill=True,
                 tree_blend=0.2, imputer_weight=1.5, random_seed=527):
        self.augment, self.tree_blend, self.imputer_weight, self.random_seed = augment, tree_blend, imputer_weight, random_seed

    # 2a: imputers trained on train + test inputs, with test-style blanks and test-mix weights
    def fit_imputer(self, df, test_features):
        joint = pd.concat([df.drop(columns='energy_usage', errors='ignore'), test_features], ignore_index=True)
        rng = np.random.default_rng(self.random_seed)
        weights = shift_weights(joint, test_features) ** self.imputer_weight
        weights = weights / weights.mean()
        self.imputation_models = {}
        for col in NUM:
            observed = joint.loc[joint[col].notna()].copy()
            copies = [observed]
            for _ in range(self.augment):
                q = observed.copy()
                mask = test_features[NUM].isna().to_numpy()[rng.integers(0, len(test_features), len(q))]
                mask[:, NUM.index(col)] = False          # the column being learned stays observed
                q[NUM] = q[NUM].mask(mask)
                copies.append(q)
            data = pd.concat(copies, ignore_index=True)
            x = cat_features(data).drop(columns=col)
            sw = np.tile(weights[observed.index.to_numpy()], self.augment + 1)
            self.imputation_models[col] = LightGBMImputer(self.random_seed).fit(x, data[col], sw)
        return self

    def fill(self, df):
        q = df.copy()
        for col, m in self.imputation_models.items():
            missing = df[col].isna()
            if missing.any():
                q.loc[missing, col] = m.predict(cat_features(df.loc[missing]).drop(columns=col))
        return q

    # 2b: Ridge on v3 features + expected previous_usage / occupancy, blended with gradient boosting
    def regression_features(self, filled, raw):
        x = linear_features(filled, self.typical).copy()
        mean_prev = self.imputation_models['previous_usage'].predict(cat_features(raw).drop(columns='previous_usage'))
        mean_occ = self.imputation_models['occupancy'].predict(cat_features(raw).drop(columns='occupancy'))
        x['expected_previous'] = mean_prev
        x['expected_occupancy'] = mean_occ
        for typ in TYPES:
            flag = (filled.building_type == typ).to_numpy(float)
            x[f'expected_previous_{typ}'] = mean_prev * flag
            x[f'expected_occupancy_{typ}'] = mean_occ * flag
        return x

    def fit(self, df, test_features):
        self.fit_imputer(df, test_features)
        data = self.fill(df)
        weekend = data.day_of_week.isin(['Saturday', 'Sunday'])
        self.typical = data.assign(is_weekend=weekend).groupby(['building_id', 'hour', 'is_weekend'])[['occupancy', 'previous_usage']].median()
        self.ridge = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 13)))
        self.ridge.fit(self.regression_features(data, df), data.energy_usage)
        self.tree = HistGradientBoostingRegressor(categorical_features='from_dtype', max_iter=1500, learning_rate=.03,
                                                  max_leaf_nodes=15, min_samples_leaf=30, l2_regularization=1.,
                                                  early_stopping=True, n_iter_no_change=50, random_state=42)
        self.tree.fit(tree_features(data), data.energy_usage - data.previous_usage)
        return self

    def predict(self, df):
        data = self.fill(df)
        a = self.ridge.predict(self.regression_features(data, df))
        b = self.tree.predict(tree_features(data)) + data.previous_usage.to_numpy()
        return (1 - self.tree_blend) * a + self.tree_blend * b

**2c. Average 10 seeds.** The random seed controls which test blank patterns go into the imputers' training copies. On the public leaderboard, the model's score moved by about 0.02 between seeds. That's luck, and it won't carry over to the hidden final-ranking rows, so averaging 10 seeds removes it. This is the slow part, roughly 15 seconds per seed.

In [6]:
inputs = test.drop(columns=['energy_usage'], errors='ignore')
runs = []
with threadpool_limits(limits=4):
    for s in SEEDS:
        runs.append(LightGBMImputedModel(**dict(NEW_MODEL_CONFIG, random_seed=s)).fit(train, inputs).predict(inputs))
runs = np.vstack(runs)
new_model = runs.mean(axis=0)
print(f'{len(SEEDS)} seeds fitted | typical difference of one seed from the average: '
      f'{np.sqrt(np.mean((runs - new_model) ** 2)):.3f}')

10 seeds fitted | typical difference of one seed from the average: 0.216


## Step 3: blend and write `candidate_final.csv`

**Weight 0.7 on the LightGBM model, 0.3 on v4.** The evidence, which covers all test rows (not just the public ones):

- Cross-validation on train with test-style blanks is best around 0.35–0.6.
- The same check, weighted to the test mix, is best around 0.75–0.8.
- Public leaderboard scores pointed to about 0.8.

At 0.7, either cross-validation check loses less than 0.002.

In [7]:
prediction = (1 - WEIGHT_NEW) * v4 + WEIGHT_NEW * new_model
assert len(prediction) == len(test) and np.isfinite(prediction).all()
result = pd.DataFrame({'id': test['id'], 'prediction': prediction})

if OUTPUT.exists():
    existing = pd.read_csv(OUTPUT)
    print(f'{OUTPUT.name} already exists; not overwritten. This run matches it to within '
          f'{np.abs(existing.prediction.to_numpy() - prediction).max():.2e}')
else:
    result.to_csv(OUTPUT, index=False)
    print(f'wrote {OUTPUT.name}')
print(f'{len(result):,} rows | columns {list(result.columns)} | blanks {result.prediction.isna().sum()} | '
      f'prediction range {result.prediction.min():.1f}-{result.prediction.max():.1f}')
result.head()

candidate_final.csv already exists; not overwritten. This run matches it to within 1.63e-10
3,000 rows | columns ['id', 'prediction'] | blanks 0 | prediction range 13.5-133.7


,id,prediction
0,EC010541,71.709319
1,EC009755,66.199118
2,EC008886,56.010157
3,EC009317,64.380118
4,EC008958,21.098596


## Results so far (public leaderboard; the final ranking uses hidden rows)

| File | Recipe | Public RMSE |
|---|---|---|
| `best_v2.csv` | MissForest + Ridge / gradient boosting | 3.40 |
| `candidate_clean.csv` | same ingredients, trained on `train.csv` only | 3.401387 |
| `candidate_v4.csv` | step 1 alone | 3.37 |
| `candidate_v5_lightgbm.csv` | 0.5 v4 + 0.5 LightGBM model (seed 527) | 3.29 |
| `candidate_middle.csv` | 0.4 v4 + 0.6 LightGBM model (3 seeds), weight from cross-validation | 3.29 |
| `candidate_v8.csv` | 0.25 v4 + 0.75 LightGBM model (seed 527) | 3.27026 |
| **`candidate_final.csv`** | **0.3 v4 + 0.7 LightGBM model (10 seeds)** | not submitted yet |

Expect a public score of about 3.28–3.29, because it doesn't get seed 527's luck. It's built to do at least as well as v8 on the hidden rows.